In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("train.csv")[["Age", "Pclass", "SibSp", "Parch", "Survived"]]


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Age       714 non-null    float64
 1   Pclass    891 non-null    int64  
 2   SibSp     891 non-null    int64  
 3   Parch     891 non-null    int64  
 4   Survived  891 non-null    int64  
dtypes: float64(1), int64(4)
memory usage: 34.9 KB


In [4]:
# count the missing values in each column
missing_values = df.isnull().sum()
print(missing_values)

Age         177
Pclass        0
SibSp         0
Parch         0
Survived      0
dtype: int64


In [5]:
# imputer
from sklearn.impute import SimpleImputer

In [6]:
imputerAge = SimpleImputer(strategy="mean")
df["Age"] = imputerAge.fit_transform(df[["Age"]]).ravel()

In [7]:
missing_values = df.isnull().sum()
print(missing_values)

Age         0
Pclass      0
SibSp       0
Parch       0
Survived    0
dtype: int64


In [8]:
X = df.drop("Survived", axis=1)
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
X.head()

,Age,Pclass,SibSp,Parch
0,22.0,3,1,0
1,38.0,1,1,0
2,26.0,3,0,0
3,35.0,1,1,0
4,35.0,3,0,0


In [10]:
y.head()

0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

In [11]:
# logistic regression and cross-validation
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

model = LogisticRegression()
scores = cross_val_score(model, X_train, y_train, cv=20)
print("Cross-validation scores:", scores)
print("Mean cross-validation score:", scores.mean())


Cross-validation scores: [0.69444444 0.75       0.69444444 0.66666667 0.63888889 0.66666667
 0.80555556 0.61111111 0.80555556 0.61111111 0.69444444 0.80555556
 0.57142857 0.77142857 0.62857143 0.71428571 0.45714286 0.77142857
 0.68571429 0.74285714]
Mean cross-validation score: 0.6893650793650794


Feature Construction


In [12]:
# i am adding the 1 to include the person himself in the family size
X['FamilySize'] = X['SibSp'] + X['Parch'] + 1
X.head()

,Age,Pclass,SibSp,Parch,FamilySize
0,22.0,3,1,0,2
1,38.0,1,1,0,2
2,26.0,3,0,0,1
3,35.0,1,1,0,2
4,35.0,3,0,0,1


In [13]:
def create_family_size(df):
    if df['FamilySize'] == 1:
        # return 'Alone'
        return 0;
    elif df['FamilySize'] <= 4:
        # return 'Small'
        return 1;
    else:
        # return 'Large'
        return 2;

In [14]:
X['FamilySizeCategory'] = X.apply(create_family_size, axis=1)

In [15]:
X.head()

,Age,Pclass,SibSp,Parch,FamilySize,FamilySizeCategory
0,22.0,3,1,0,2,1
1,38.0,1,1,0,2,1
2,26.0,3,0,0,1,0
3,35.0,1,1,0,2,1
4,35.0,3,0,0,1,0


In [16]:
X.drop(['SibSp', 'Parch'], axis=1, inplace=True)

In [17]:
X.head()

,Age,Pclass,FamilySize,FamilySizeCategory
0,22.0,3,2,1
1,38.0,1,2,1
2,26.0,3,1,0
3,35.0,1,2,1
4,35.0,3,1,0


In [18]:
X_train_2, X_test_, y_train_2, y_test_2 = train_test_split(X, y, test_size=0.2, random_state=42)

In [19]:
# again cross validation with the new features
model_2 = LogisticRegression()
scores_2 = cross_val_score(model_2, X_train_2, y_train_2, cv=20)
print("Cross-validation scores with new features:", scores_2)
print("Mean cross-validation score with new features:", scores_2.mean())

Cross-validation scores with new features: [0.69444444 0.77777778 0.72222222 0.63888889 0.66666667 0.72222222
 0.75       0.58333333 0.80555556 0.61111111 0.72222222 0.72222222
 0.6        0.77142857 0.62857143 0.77142857 0.48571429 0.74285714
 0.71428571 0.68571429]
Mean cross-validation score with new features: 0.6908333333333333


Feature Splitting


In [20]:
df_2  = pd.read_csv("train.csv")

In [21]:
name = df_2["Name"][11]
print(name)

last_Name = name.split(",")[0].strip()
print(last_Name)

salutation = name.split(",")[1].split(".")[0].strip()
print(salutation)

first_name = name.split(",")[1].split(".")[1].strip()
print(first_name)

Bonnell, Miss. Elizabeth
Bonnell
Miss
Elizabeth


In [22]:
# create new columns for last name, salutation, and first name with simply
df_2["LastName"] = df_2["Name"].apply(lambda x: x.split(",")[0].strip())

df_2["Salutation"] = df_2["Name"].apply(lambda x: x.split(",")[1].split(".")[0].strip())

df_2["FirstName"] = df_2["Name"].apply(lambda x: x.split(",")[1].split(".")[1].strip())

In [23]:
df_2.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,LastName,Salutation,FirstName
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Braund,Mr,Owen Harris
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Cumings,Mrs,John Bradley (Florence Briggs Thayer)
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Heikkinen,Miss,Laina
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Futrelle,Mrs,Jacques Heath (Lily May Peel)
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Allen,Mr,William Henry


In [24]:
df_2.drop(['Name'], axis=1, inplace=True)

In [25]:
df_2.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,LastName,Salutation,FirstName
0,1,0,3,male,22.0,1,0,A/5 21171,7.2500,NaN,S,Braund,Mr,Owen Harris
1,2,1,1,female,38.0,1,0,PC 17599,71.2833,C85,C,Cumings,Mrs,John Bradley (Florence Briggs Thayer)
2,3,1,3,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Heikkinen,Miss,Laina
3,4,1,1,female,35.0,1,0,113803,53.1000,C123,S,Futrelle,Mrs,Jacques Heath (Lily May Peel)
4,5,0,3,male,35.0,0,0,373450,8.0500,NaN,S,Allen,Mr,William Henry
